# Code-Driven Pipeline (Inline)

This notebook demonstrates building clinical datasets entirely in Python code. It is best suited for quick ad-hoc analysis, smaller studies, or initial data discovery.

In [ ]:
import polars as pl
import fleet

# Create some raw mock data
raw_dm = pl.DataFrame({
    "usubjid": ["S001", "S002"],
    "trt": ["Drug A", "Placebo"],
    "age": [35, 42]
})

raw_vs = pl.DataFrame({
    "usubjid": ["S001", "S001", "S002"],
    "test": ["SysBP", "DiaBP", "SysBP"],
    "val": [120, 80, 130]
})

## 1. Derive SDTM

We use the `.sdtm` namespace to standardize the raw inputs.

In [ ]:
dm = (
    raw_dm.sdtm
    .hardcode_no_ct("DOMAIN", value="DM")
    .assign_no_ct("USUBJID", source="usubjid")
    .assign_no_ct("ARM", source="trt")
    .assign_no_ct("AGE", source="age")
    .to_frame()
)

vs = (
    raw_vs.sdtm
    .hardcode_no_ct("DOMAIN", value="VS")
    .assign_no_ct("USUBJID", source="usubjid")
    .assign_no_ct("VSTESTCD", source="test")
    .assign_no_ct("VSSTRESN", source="val")
    .to_frame()
)

print(dm)
print(vs)

## 2. Derive ADaM

We use the `.adam` namespace. Notice how easily we define a custom derivation to flag subjects who are older than 40.

In [ ]:
adsl = (
    dm.adam.adsl
    .source("USUBJID")
    .source("ARM")
    .source("AGE")
    # Custom inline derivation not covered by standard library verbs
    .derivation("AGE40FL", expression=pl.when(pl.col("AGE") > 40).then(pl.lit("Y")).otherwise(pl.lit("N")))
    .to_frame()
)

print(adsl)